# Defects and Materials Modeling

This notebook introduces practical atomistic modeling tasks for crystalline materials. The focus is on building a scientific workflow: create a structure, introduce a defect or loading condition, relax or simulate it in LAMMPS, and analyze it in OVITO.

## 1. Example Modeling Questions

Materials MD simulations often begin with questions like:

- What is the formation energy of a vacancy?
- How does a surface reconstruct after relaxation?
- How does a crystal respond to tensile strain?
- When does a solid melt during heating?
- How do dislocations or grain boundaries evolve during deformation?

## 2. Vacancy Formation Energy

A common introductory defect calculation is vacancy formation energy:

$$E_f^v = E_{N-1}^{vac} - \frac{N-1}{N}E_N^{bulk}$$

where `E_N_bulk` is the energy of the perfect crystal with `N` atoms and `E_vac` is the energy of the relaxed structure with one atom removed.

In [ ]:
def vacancy_formation_energy(e_bulk, n_bulk, e_vac, n_vac):
    expected_n_vac = n_bulk - 1
    if n_vac != expected_n_vac:
        raise ValueError(f"Expected {expected_n_vac} atoms in vacancy cell, got {n_vac}.")
    return e_vac - (n_vac / n_bulk) * e_bulk


# Example placeholder values in eV. Replace these with relaxed LAMMPS energies.
e_bulk = -3400.0
n_bulk = 1000
e_vac = -3395.8
n_vac = 999

ef_vac = vacancy_formation_energy(e_bulk, n_bulk, e_vac, n_vac)
print(f"Vacancy formation energy: {ef_vac:.3f} eV")

In [ ]:
from pathlib import Path

for dirname in ["atomsk", "lammps", "ovito", "data", "figures"]:
    Path(dirname).mkdir(exist_ok=True)

workflow = """Defect modeling workflow

1. Build a perfect crystal using Atomsk or LAMMPS.
2. Relax the perfect crystal and record total energy and atom count.
3. Remove one atom to create a vacancy.
4. Relax the vacancy structure with the same potential and settings.
5. Compute vacancy formation energy.
6. Visualize local relaxation around the defect in OVITO.
"""

Path("data/defect_modeling_workflow.txt").write_text(workflow)
print(workflow)

## 3. LAMMPS Skeleton for Relaxation

The following template can be adapted for both the perfect and vacancy structures. Replace the potential file and material names with values appropriate for the system being modeled.

In [ ]:
relax_script = """# Generic relaxation template
units           metal
dimension       3
boundary        p p p
atom_style      atomic

read_data       data/input_structure.lmp

pair_style      eam/alloy
pair_coeff      * * potential.eam.alloy Element

thermo          100
thermo_style    custom step pe etotal press pxx pyy pzz

min_style       cg
minimize        1.0e-10 1.0e-12 1000 10000

write_data      data/relaxed_structure.lmp
"""

Path("lammps/in.relax_template").write_text(relax_script)
print(relax_script)

## Exercises

1. Build a perfect FCC crystal and relax it.
2. Create one vacancy and relax it.
3. Compute vacancy formation energy.
4. Visualize the local relaxed defect environment in OVITO.
5. Repeat for a larger cell and compare finite-size effects.